In [1]:
import numpy as np
import pandas as pd
import random

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ----------------- Risk Assignment Function -----------------
def assign_risk(hr, spo2, sleep, stress, temp, age, systolic_bp, diastolic_bp):
    # Determine BP label first
    if systolic_bp >= 140 or diastolic_bp >= 90:
        bp_risk = "High_BP"
    elif systolic_bp < 90 or diastolic_bp < 60:
        bp_risk = "Low_BP"
    else:
        bp_risk = "Normal_BP"

    # Priority order: Cardiac > Fever > Respiratory > Stress > Normal
    # Cardiac Risk conditions
    if (hr > 110 and spo2 < 93) or bp_risk == "High_BP" or age > 60:
        return "Cardiac_Risk"

    # Respiratory Risk
    elif spo2 < 94:
        return "Respiratory_Risk"

    # Fever Risk
    elif temp >= 38.5:
        return "Fever_Risk"

    # Stress Risk
    elif stress > 7 and sleep < 5:
        return "Stress_Risk"

    # Normal
    else:
        return "Normal"

# ----------------- Dataset Generation -----------------
data = []

for _ in range(6000):
    heart_rate = random.randint(55, 130)
    spo2 = random.randint(90, 100)
    sleep_hours = round(random.uniform(3, 9), 1)
    stress_level = random.randint(0, 10)
    body_temp = round(random.uniform(35.8, 39.5), 1)
    age = random.randint(18, 80)
    systolic_bp = random.randint(90, 160)    # realistic range
    diastolic_bp = random.randint(60, 100)   # realistic range

    # ✅ Pass all 8 parameters to assign_risk
    risk = assign_risk(
        heart_rate,
        spo2,
        sleep_hours,
        stress_level,
        body_temp,
        age,
        systolic_bp,
        diastolic_bp
    )

    data.append([
        heart_rate,
        spo2,
        sleep_hours,
        stress_level,
        body_temp,
        age,
        systolic_bp,
        diastolic_bp,
        risk
    ])

# ----------------- Convert to DataFrame -----------------
columns = [
    "heart_rate",
    "spo2",
    "sleep_hours",
    "stress_level",
    "body_temp",
    "age",
    "systolic_bp",
    "diastolic_bp",
    "risk_category"
]

df = pd.DataFrame(data, columns=columns)

# Save to CSV
df.to_csv("healthsense_synthetic_dataset.csv", index=False)

print(df.head())

   heart_rate  spo2  sleep_hours  stress_level  body_temp  age  systolic_bp  \
0          81    98          6.7             2       39.3   28          110   
1         119    97          8.7             1       38.6   44           93   
2         106    92          5.7             3       39.2   45          160   
3          80    91          6.4             8       37.9   27          146   
4          93    92          5.7             2       38.3   24          124   

   diastolic_bp     risk_category  
0            76        Fever_Risk  
1            99      Cardiac_Risk  
2            81      Cardiac_Risk  
3            98      Cardiac_Risk  
4            86  Respiratory_Risk  


In [2]:
df

,heart_rate,spo2,sleep_hours,stress_level,body_temp,age,systolic_bp,diastolic_bp,risk_category
0,81,98,6.7,2,39.3,28,110,76,Fever_Risk
1,119,97,8.7,1,38.6,44,93,99,Cardiac_Risk
2,106,92,5.7,3,39.2,45,160,81,Cardiac_Risk
3,80,91,6.4,8,37.9,27,146,98,Cardiac_Risk
4,93,92,5.7,2,38.3,24,124,86,Respiratory_Risk
...,...,...,...,...,...,...,...,...,...
5995,122,98,8.3,4,36.8,18,99,61,Normal
5996,95,90,3.8,3,36.7,18,96,71,Respiratory_Risk
5997,70,96,4.5,8,36.8,40,151,68,Cardiac_Risk
5998,97,98,7.3,8,36.3,69,131,68,Cardiac_Risk


In [3]:
X = df.drop("risk_category", axis=1)
y = df["risk_category"]


In [4]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)


Classes: ['Cardiac_Risk' 'Fever_Risk' 'Normal' 'Respiratory_Risk' 'Stress_Risk']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded,test_size=0.2,random_state=42)


In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [7]:

model = RandomForestClassifier(
    n_estimators=80,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42
)

model.fit(X_train_scaled, y_train)


RandomForestClassifier(max_depth=5, min_samples_leaf=10, n_estimators=80,
                       random_state=42)

In [8]:
y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_
))


Accuracy: 0.975
                  precision    recall  f1-score   support

    Cardiac_Risk       0.99      1.00      0.99       805
      Fever_Risk       1.00      1.00      1.00        66
          Normal       0.91      1.00      0.95       191
Respiratory_Risk       1.00      0.92      0.96       118
     Stress_Risk       0.00      0.00      0.00        20

        accuracy                           0.97      1200
       macro avg       0.78      0.78      0.78      1200
    weighted avg       0.96      0.97      0.97      1200



C:\Users\kanik\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\kanik\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\kanik\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
sample_input = pd.DataFrame([{
    "heart_rate": 115,
    "spo2": 92,
    "sleep_hours": 4.5,
    "stress_level": 8,
    "body_temp": 38.8,
    "age": 40,
    "systolic_bp": 145,
    "diastolic_bp": 95
}])

# Scale using the same scaler
sample_scaled = scaler.transform(sample_input)

# Predict
prediction = model.predict(sample_scaled)
predicted_risk = label_encoder.inverse_transform(prediction)[0]

print("Predicted Risk:", predicted_risk)

Predicted Risk: Cardiac_Risk


In [10]:
import joblib

In [11]:
joblib.dump(model, "healthsense_model_final.pkl")

['healthsense_model_final.pkl']

In [12]:


# These variables must match the names you used in your notebook
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Files saved successfully!")

Files saved successfully!
